# 提示词模板（Prompt Templates）
在langchain中，构造提示词既可以直接调用python字符串拼接

也可以直接使用langchain提供的提示词模板类，如`PromptTemplate`、`ChatPromptTemplate`等。

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
import os

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

举例1：

In [ ]:
#字符串拼接
topic = "python"
difficulty = "中级"

# 难以维护，容易出错
prompt_str = f"你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。"
response = model.invoke(prompt_str)
print(f"AI 回复：{response.content}...\n")

可以看出缺点来可读性差，并且难以维护

容易出错，难以支撑负责场景

举例2：

In [ ]:
# PromptTemplate
from langchain.prompts import PromptTemplate 

topic = "python"
difficulty = "中级"

template = PromptTemplate.from_template("你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。")

# 使用提示词模板
prompt = template.format(topic = topic, difficulty = difficulty)

response = model.invoke(prompt)
print(f"AI 回复：{response.content}...\n")

In [ ]:
# ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate(
    [
        {"system", "你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。"},
        {"human", "{input}"}
    ]
)

# 调用
prompt = prompt_template.invoke({"name":"小Tan导师","user_input":"我能帮你什么忙嘛？"})

print(prompt)

可以看到这种 结构清晰 容易维护，并且支持复杂的场景 比如对话 RAG Agent 

现在的机制主要是以ChatModel + ChatPromptTemplate为主

现代聊天模型 API 已原生 支持角色概念 。它们不再接受单一字符串，

而是要求输入 一个结构化的消息列表 。为构建复杂、可靠的多轮对话智能体系统奠定了坚实的基础。

*Prompt 工具*： ChatPromptTemplate 因此成为LangChain 1.0 中最核心的 Prompt工具。它的职
责是接收变量，并输出一个 List「BaseMessage］（消息列表），该列表可直接传递给聊天模型。

## ChatPromptTemplate的使用


chatpromptTemplate是用于生成消息列表的核心组件

ChatPromptTemplate创建消息列表的提示词模板，ta比普通PromptTemplate更加强大,支持System/user/assistant等不同的角色

对比：

ChatPromptTemplate 可以通过 初始化方法 或 from_messages 方法来实例化提示词模板。实例化
时需要传入 messages参数 。常见类型是：tuple构成的列表，参数类型（role : str，content : str ）

方式1(推荐)：调用from_messages()：

该方法允许传入一个由元组（Tuple）构成的列表，列表中的每一个元组都代表一条具有特定角色的消
息。

举例1：

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

#定义聊天提示词模板
chat_template = ChatPromptTemplate.format_messages(
    [
        "system","你是一个有帮助的AI机器人，你的名字叫{name}"
        "human","你好！最近怎么样了，过的还好嘛？"
        "ai","我很好，谢谢！"
        "human","{user_input}"
    ]
)

#格式化聊天提示词模板中的变量
prompt = chat_template.invoke({"name":"小谷","user_input":"你好！"})

# 打印
print(prompt)
